# 金融对话数值推理模型：SFT → 轻量 DPO / GRPO → benchmark 评估

目标：训练一个面向 **金融对话数值推理** 的 reasoning model。

整体闭环：
- **SFT-1 主干推理**：`ConvFinQA + FinQA`
- **SFT-2 中文补强**：`fingpt-fineval + 少量 fingpt-fiqa_qa`
- **DPO（可选）**：小规模优化表达质量、结构和少废话
- **GRPO（推荐）**：基于可验证 reward 优化答案正确性、程序一致性与结构约束
- **benchmark 评估**：`FinQA / ConvFinQA + CFLUE + FinanceBench + AdaptLLM/finance-tasks`

本 Notebook 复用 MedicalGPT 的训练框架：
- 数据格式遵循 `docs/datasets.md`
- pipeline 参考 `run_training_dpo_pipeline.ipynb`


## 0. 环境准备（可选）

如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [2]:
!pip install -r requirements.txt --upgrade

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 32.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 11.7 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 9.7 MB/s  0:00:00m eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 10.3 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 14.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 13.0 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 10.4 MB/s  0:00:01 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 58.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 9.6 MB/s  0:00:540:00:0100:02
     ━━━━━━━

In [1]:
!pip install modelscope


Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 10.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [modelscope]8 [modelscope]


In [3]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


HF_ENDPOINT: https://hf-mirror.com


## 1. 任务配置

这里把配置拆成五部分：
1. 主干推理数据 `SFT-1`
2. 中文补强数据 `SFT-2`
3. 原始数据下载缓存目录
4. 转换 / 清洗 / 混合目录
5. 训练与评估输出目录


In [19]:
BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct"

In [43]:
from pathlib import Path
import json
import random
from itertools import islice


TEMPLATE_NAME = "qwen"
RANDOM_SEED = 42
FORCE_REDOWNLOAD_RAW = False

# ==== 根路径统一配置（数据盘） ====
DISK_ROOT = Path("/root/autodl-tmp")
OUT_DIR = DISK_ROOT / "data" / "financial_reasoning"
OUTPUT_ROOT = DISK_ROOT / "outputs" / "financial_reasoning"

RAW_CACHE_DIR = OUT_DIR / "raw"
SFT1_SFT_DIR = OUT_DIR / "sft1_sharegpt"
SFT2_SFT_DIR = OUT_DIR / "sft2_sharegpt"
DPO_DIR = OUT_DIR / "dpo_pairs"
MIXED_DIR = OUT_DIR / "mixed"
CLEAN_DIR = OUT_DIR / "clean"
REPORT_DIR = OUT_DIR / "reports"

SFT1_MIXED_FILE = MIXED_DIR / "train_sft1_mixed.jsonl"
SFT2_MIXED_FILE = MIXED_DIR / "train_sft2_mixed.jsonl"
DPO_MIXED_FILE = DPO_DIR / "train_reasoning_dpo.jsonl"
SFT1_CLEAN_FILE = CLEAN_DIR / "train_sft1_clean.jsonl"
SFT2_CLEAN_FILE = CLEAN_DIR / "train_sft2_clean.jsonl"

SFT1_DIR = CLEAN_DIR / "sft1_dir_strict"
SFT2_DIR = CLEAN_DIR / "sft2_dir_strict"
DPO_TRAIN_DIR = DPO_DIR / "train_dir"

SFT1_AUDIT_DIR = CLEAN_DIR / "audit_sft1"
SFT2_AUDIT_DIR = CLEAN_DIR / "audit_sft2"
SFT1_STRICT_FILE = CLEAN_DIR / "train_sft1_clean_strict.jsonl"
SFT2_STRICT_FILE = CLEAN_DIR / "train_sft2_clean_strict.jsonl"

# 训练/日志输出改到数据盘
SFT1_OUT = OUTPUT_ROOT / "sft1_lora"
SFT1_MERGED_OUT = OUTPUT_ROOT / "sft1_merged"
SFT2_OUT = OUTPUT_ROOT / "sft2_lora"
SFT2_MERGED_OUT = OUTPUT_ROOT / "sft2_merged"
DPO_OUT = OUTPUT_ROOT / "dpo_lora"
DPO_MERGED_OUT = OUTPUT_ROOT / "dpo_merged"
TB_LOG_DIR = OUTPUT_ROOT / "tensorboard"

for d in [
    RAW_CACHE_DIR, SFT1_SFT_DIR, SFT2_SFT_DIR, DPO_DIR, MIXED_DIR, CLEAN_DIR, REPORT_DIR,
    SFT1_DIR, SFT2_DIR, DPO_TRAIN_DIR, SFT1_AUDIT_DIR, SFT2_AUDIT_DIR, OUTPUT_ROOT, TB_LOG_DIR,
    SFT1_OUT, SFT1_MERGED_OUT, SFT2_OUT, SFT2_MERGED_OUT, DPO_OUT, DPO_MERGED_OUT,
]:
    d.mkdir(parents=True, exist_ok=True)

SFT1_TOTAL_BUDGET = 12000
SFT2_EXTRA_BUDGET = 2500
DPO_TOTAL_BUDGET = 4000
MAX_DPO_PER_DATASET = 2000

# SFT2: Fingpt为主体 + 清洗后的SFT1 replay
SFT2_REPLAY_RATIO = 0.2   # replay 行数 = fingpt主体行数 * ratio
SFT2_REPLAY_MAX_ROWS = 3000

SFT1_DATA_SPECS = [
    {
        "name": "fingpt_convfinqa_train",
        "family": "convfinqa_turn",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "convfinqa_turn" / "train_turn.json",
        "split": "train",
        "weight": 1.0,
        "max_rows": None,
        "description": "多轮金融对话 + 数值推理主干数据（使用本地已上传文件）",
    },
    {
        "name": "finqa_train",
        "family": "finqa",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "finqa" / "train.json",
        "split": "train",
        "weight": 0.7,
        "max_rows": 6251,
        "description": "表格 + 文本 + 程序推理主干数据（使用本地已上传文件）",
    },
]

SFT2_DATA_SPECS = [
    {
        "name": "fingpt_fineval_train",
        "family": "fineval",
        "source_type": "hf",
        "dataset_name": "FinGPT/fingpt-fineval",
        "split": "train",
        "weight": 1.0,
        "max_rows": 1060,
        "description": "中文金融考试推理补强",
    },
    {
        "name": "fingpt_fiqa_qa_train",
        "family": "fiqa_qa",
        "source_type": "hf",
        "dataset_name": "FinGPT/fingpt-fiqa_qa",
        "split": "train",
        "weight": 0.35,
        "max_rows": 1500,
        "description": "少量中文/对话式金融 QA 补强",
    },
]

BENCHMARK_SPECS = [
    {"name": "ConvFinQA_dev", "role": "核心多轮数值推理"},
    {"name": "FinQA_dev", "role": "核心表文混合推理"},
    {"name": "CFLUE", "role": "中文金融泛化"},
    {"name": "FinanceBench", "role": "开放书金融 QA 迁移"},
    {"name": "AdaptLLM/finance-tasks", "role": "补充 benchmark"},
]


def allocate_target_rows(specs, total_budget: int, default_cap: int | None = None):
    if not specs:
        return []
    total_weight = sum(max(float(spec.get("weight", 0.0)), 0.0) for spec in specs)
    if total_weight <= 0:
        raise ValueError("All dataset weights are zero; cannot allocate target_rows.")

    allocated = []
    for spec in specs:
        raw_target = int(round(total_budget * float(spec.get("weight", 0.0)) / total_weight))
        cap = spec.get("max_rows", default_cap)
        if cap is not None:
            raw_target = min(raw_target, int(cap))
        spec = dict(spec)
        spec["target_rows"] = max(raw_target, 0)
        allocated.append(spec)
    return allocated


SFT1_DATA_SPECS = allocate_target_rows(SFT1_DATA_SPECS, SFT1_TOTAL_BUDGET)
SFT2_DATA_SPECS = allocate_target_rows(SFT2_DATA_SPECS, SFT2_EXTRA_BUDGET)

print("DISK_ROOT:", DISK_ROOT)
print("OUT_DIR:", OUT_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("BASE_MODEL:", BASE_MODEL)
print("SFT1_TOTAL_BUDGET:", SFT1_TOTAL_BUDGET)
print("SFT2_EXTRA_BUDGET:", SFT2_EXTRA_BUDGET)
print("SFT2_REPLAY_RATIO:", SFT2_REPLAY_RATIO)
print("DPO_TOTAL_BUDGET:", DPO_TOTAL_BUDGET)
print("SFT1_DATA_SPECS:", json.dumps([{**spec, 'local_path': str(spec.get('local_path', ''))} for spec in SFT1_DATA_SPECS], ensure_ascii=False, indent=2))
print("SFT2_DATA_SPECS:", json.dumps(SFT2_DATA_SPECS, ensure_ascii=False, indent=2))
print("BENCHMARK_SPECS:", json.dumps(BENCHMARK_SPECS, ensure_ascii=False, indent=2))


DISK_ROOT: /root/autodl-tmp
OUT_DIR: /root/autodl-tmp/data/financial_reasoning
OUTPUT_ROOT: /root/autodl-tmp/outputs/financial_reasoning
BASE_MODEL: /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct
SFT1_TOTAL_BUDGET: 12000
SFT2_EXTRA_BUDGET: 2500
SFT2_REPLAY_RATIO: 0.2
DPO_TOTAL_BUDGET: 4000
SFT1_DATA_SPECS: [
  {
    "name": "fingpt_convfinqa_train",
    "family": "convfinqa_turn",
    "source_type": "local",
    "local_path": "/root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json",
    "split": "train",
    "weight": 1.0,
    "max_rows": null,
    "description": "多轮金融对话 + 数值推理主干数据（使用本地已上传文件）",
    "target_rows": 7059
  },
  {
    "name": "finqa_train",
    "family": "finqa",
    "source_type": "local",
    "local_path": "/root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json",
    "split": "train",
    "weight": 0.7,
    "max_rows": 6251,
    "description": "表格 + 文本 + 程序推理主干数据（使用本地已上传文件）",
    "target_rows": 4941
  }
]
SFT2_DATA_SPECS: [
  {
    "name

## 1.1 统一任务格式

推荐训练模板：
- **ConvFinQA**：历史对话 + 当前问题 + 表格/文本上下文 → 分步推理 → 最终答案
- **FinQA**：文本 + 表格 + 问题 → reasoning program / 中间步骤 → 执行结果
- **fingpt-fineval**：题目 + 选项 → 简短推理解释 → 正确选项
- **fingpt-fiqa_qa**：问题 → 结构化解释 → 结论

MedicalGPT 对数据格式的要求：
- **SFT**：`conversations`
- **DPO**：`question + response_chosen + response_rejected`


In [24]:
from pprint import pprint

SFT_SCHEMA_EXAMPLE = {
    "conversations": [
        {"from": "human", "value": "历史对话 + 当前问题 + 文本/表格上下文"},
        {"from": "gpt", "value": "问题分析...\n关键证据...\n推理程序...\n最终答案..."},
    ]
}

DPO_SCHEMA_EXAMPLE = {
    "system": "",
    "history": [],
    "question": "统一格式的金融推理问题",
    "response_chosen": "结构化且正确的回答",
    "response_rejected": "答案错误或程序不一致的回答",
}

print("[MedicalGPT SFT schema]")
pprint(SFT_SCHEMA_EXAMPLE, sort_dicts=False)
print("[MedicalGPT DPO schema]")
pprint(DPO_SCHEMA_EXAMPLE, sort_dicts=False)


[MedicalGPT SFT schema]
{'conversations': [{'from': 'human', 'value': '历史对话 + 当前问题 + 文本/表格上下文'},
                   {'from': 'gpt',
                    'value': '问题分析...\n关键证据...\n推理程序...\n最终答案...'}]}
[MedicalGPT DPO schema]
{'system': '',
 'history': [],
 'question': '统一格式的金融推理问题',
 'response_chosen': '结构化且正确的回答',
 'response_rejected': '答案错误或程序不一致的回答'}


In [25]:
print("[Training stages]")
print({
    "SFT-1": [spec["name"] for spec in SFT1_DATA_SPECS],
    "SFT-2": [spec["name"] for spec in SFT2_DATA_SPECS],
    "DPO": "从上述 raw 数据自动构造轻量偏好对",
    "GRPO": "基于格式奖励 + 程序一致性 + 答案正确性",
})


[Training stages]
{'SFT-1': ['fingpt_convfinqa_train', 'finqa_train'], 'SFT-2': ['fingpt_fineval_train', 'fingpt_fiqa_qa_train'], 'DPO': '从上述 raw 数据自动构造轻量偏好对', 'GRPO': '基于格式奖励 + 程序一致性 + 答案正确性'}


## 2. 下载原始数据到本地缓存

这一阶段只负责下载 raw 数据，避免每次重跑 notebook 都重复下载。
- `url_json`：直接下载官方 JSON 文件
- `hf`：通过 `datasets.load_dataset` 落地到本地 jsonl
- 若缓存已存在且 `FORCE_REDOWNLOAD_RAW=False`，则直接跳过


In [26]:
import os

for k in ["HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"]:
    print(k, "=", os.environ.get(k))

HTTP_PROXY = None
HTTPS_PROXY = None
ALL_PROXY = None
http_proxy = None
https_proxy = None
all_proxy = None


In [27]:
!pip install -U datasets huggingface_hub httpx

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [28]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"

In [29]:
import os, requests

print("HTTP_PROXY =", os.environ.get("HTTP_PROXY"))
print("HTTPS_PROXY =", os.environ.get("HTTPS_PROXY"))
print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))

url = "https://huggingface.co/api/datasets/FinGPT/fingpt-convfinqa"
r = requests.get(url, timeout=30)
print(r.status_code)
print(r.text[:500])

HTTP_PROXY = http://127.0.0.1:7890
HTTPS_PROXY = http://127.0.0.1:7890
HF_ENDPOINT = None
200
{"_id":"6524f11d9d870c61b310d76e","id":"FinGPT/fingpt-convfinqa","author":"FinGPT","sha":"130ed6276b6ba5cc188eb4eafa558f3312f5bc4d","lastModified":"2023-10-10T06:44:37.000Z","private":false,"gated":false,"disabled":false,"tags":["size_categories:10K<n<100K","format:parquet","modality:text","library:datasets","library:pandas","library:mlcroissant","library:polars","region:us"],"description":"\n\t\n\t\t\n\t\tDataset Card for \"fingpt-convfinqa\"\n\t\n\nMore Information needed\n","downloads":653,"l


In [30]:
import json
from datasets import load_dataset

ALL_DATA_SPECS = SFT1_DATA_SPECS + SFT2_DATA_SPECS


def raw_cache_file(spec: dict) -> Path:
    if spec["source_type"] == "local":
        return Path(spec["local_path"])
    return RAW_CACHE_DIR / spec["family"] / f"{spec['name']}.jsonl"

raw_files = {}
for spec in ALL_DATA_SPECS:
    cache_file = raw_cache_file(spec)
    raw_files[spec["name"]] = cache_file
    cache_file.parent.mkdir(parents=True, exist_ok=True)

    if spec["source_type"] == "local":
        if not cache_file.exists():
            raise FileNotFoundError(f"Missing local raw file: {cache_file}")
        print(f"[use local] {spec['name']} -> {cache_file}")
        continue

    if cache_file.exists() and not FORCE_REDOWNLOAD_RAW:
        print(f"[skip] use cached raw file: {cache_file}")
        continue

    print(f"[download] {spec['name']} -> {cache_file}")
    if spec["source_type"] == "hf":
        ds = load_dataset(spec["dataset_name"], split=spec["split"])
        with cache_file.open('w', encoding='utf-8') as f:
            for row in ds:
                f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")
    else:
        raise ValueError(f"Unsupported source_type: {spec['source_type']}")
    print(f"[saved] {cache_file}")


/root/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[use local] fingpt_convfinqa_train -> /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json
[use local] finqa_train -> /root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json
[skip] use cached raw file: /root/autodl-tmp/data/financial_reasoning/raw/fineval/fingpt_fineval_train.jsonl
[skip] use cached raw file: /root/autodl-tmp/data/financial_reasoning/raw/fiqa_qa/fingpt_fiqa_qa_train.jsonl


## 2.1 展示原始数据前几条案例

按你的要求，这里直接展示主干数据集 `ConvFinQA / FinQA` 的前几条 raw 样例。

说明：
- `ConvFinQA` 直接读取你已上传的本地文件：`data/financial_reasoning/raw/convfinqa_turn/train_turn.json`
- `FinQA` 直接读取你已上传的本地文件：`data/financial_reasoning/raw/finqa/train.json`


In [31]:
def show_first_records(path: Path, n: int = 2):
    print(f"\n=== {path} ===")
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        for obj in data[:n]:
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:3000])
    else:
        print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])

print(SFT1_DATA_SPECS)
for spec in SFT1_DATA_SPECS:
    show_first_records(raw_files[spec["name"]], n=2)



[{'name': 'fingpt_convfinqa_train', 'family': 'convfinqa_turn', 'source_type': 'local', 'local_path': PosixPath('/root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json'), 'split': 'train', 'weight': 1.0, 'max_rows': None, 'description': '多轮金融对话 + 数值推理主干数据（使用本地已上传文件）', 'target_rows': 7059}, {'name': 'finqa_train', 'family': 'finqa', 'source_type': 'local', 'local_path': PosixPath('/root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json'), 'split': 'train', 'weight': 0.7, 'max_rows': 6251, 'description': '表格 + 文本 + 程序推理主干数据（使用本地已上传文件）', 'target_rows': 4941}]

=== /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json ===
{
  "pre_text": [
    "26 | 2009 annual report in fiscal 2008 , revenues in the credit union systems and services business segment increased 14% ( 14 % ) from fiscal 2007 .",
    "all revenue components within the segment experienced growth during fiscal 2008 .",
    "license revenue generated the largest dollar growth i

## 3. 转换格式：raw → MedicalGPT SFT / DPO

这一阶段只读取本地 raw 文件：
- `fin_to_sharegpt.py`：转换为 SFT 格式
- `fin_to_dpo_pairs.py`：转换为轻量 DPO 格式


In [32]:
import subprocess

sft_reports = []
dpo_reports = []

for spec in ALL_DATA_SPECS:
    raw_file = raw_files[spec["name"]]
    out_dir = SFT1_SFT_DIR if spec in SFT1_DATA_SPECS else SFT2_SFT_DIR
    sft_file = out_dir / f"{spec['name']}_sharegpt.jsonl"
    dpo_file = DPO_DIR / f"{spec['name']}_dpo.jsonl"

    sharegpt_cmd = [
        "python", "fin_to_sharegpt.py",
        "--source_file", str(raw_file),
        "--output_file", str(sft_file),
        "--dataset_family", spec["family"],
    ]
    dpo_cmd = [
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(raw_file),
        "--output_file", str(dpo_file),
        "--dataset_family", spec["family"],
        "--seed", str(RANDOM_SEED),
    ]

    print(" ".join(sharegpt_cmd))
    sharegpt_result = subprocess.run(sharegpt_cmd, check=True, capture_output=True, text=True)
    print(sharegpt_result.stdout)
    sft_reports.append(json.loads(sharegpt_result.stdout))

    print(" ".join(dpo_cmd))
    dpo_result = subprocess.run(dpo_cmd, check=True, capture_output=True, text=True)
    print(dpo_result.stdout)
    dpo_reports.append(json.loads(dpo_result.stdout))

print("[SFT conversion reports]")
print(json.dumps(sft_reports, ensure_ascii=False, indent=2))
print("[DPO conversion reports]")
print(json.dumps(dpo_reports, ensure_ascii=False, indent=2))


python fin_to_sharegpt.py --source_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json --output_file /root/autodl-tmp/data/financial_reasoning/sft1_sharegpt/fingpt_convfinqa_train_sharegpt.jsonl --dataset_family convfinqa_turn
{
  "output_file": "/root/autodl-tmp/data/financial_reasoning/sft1_sharegpt/fingpt_convfinqa_train_sharegpt.jsonl",
  "dataset_family": "convfinqa_turn",
  "saved_rows": 7462,
  "skipped_rows": 3642
}

python fin_to_dpo_pairs.py --source_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json --output_file /root/autodl-tmp/data/financial_reasoning/dpo_pairs/fingpt_convfinqa_train_dpo.jsonl --dataset_family convfinqa_turn --seed 42
{
  "output_file": "/root/autodl-tmp/data/financial_reasoning/dpo_pairs/fingpt_convfinqa_train_dpo.jsonl",
  "dataset_family": "convfinqa_turn",
  "saved_rows": 7462,
  "skipped_rows": 3642
}

python fin_to_sharegpt.py --source_file /root/autodl-tmp/data/financial_reasoning/raw/fin

## 3.1 展示转换后的前几条案例

按你的要求，这里展示转换后的 `ConvFinQA / FinQA` SFT 样例，确认模板、表格上下文、历史对话和推理程序是否正常。


In [34]:
for spec in SFT1_DATA_SPECS:
    sft_file = SFT1_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
    print(f"\n=== {spec['name']} converted sample ===")
    with sft_file.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 2:
                break
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2)[:3500])



=== fingpt_convfinqa_train converted sample ===
{
  "source_dataset": "ConvFinQA",
  "task_type": "financial_conversational_numerical_reasoning",
  "record_id": "Single_JKHY/2009/page_28.pdf-3_0",
  "metadata": {
    "turn_ind": 0,
    "cur_type": "number_turn",
    "program": "divide(subtract(206588, 181001), 181001)",
    "gold_ind": {
      "table_6": "2008 the net cash from operating activities of year ended june 30 2009 2008 is $ 206588 ; the net cash from operating activities of year ended june 30 2009 2008 is $ 181001 ; the net cash from operating activities of year ended june 30 2009 is $ 174247 ;"
    }
  },
  "conversations": [
    {
      "from": "human",
      "value": "你是一名金融数值推理助手。请结合对话历史、文本材料和表格，进行分步推理并给出最终数值答案。\n\n材料（表格前文本）：\n- 26 | 2009 annual report in fiscal 2008 , revenues in the credit union systems and services business segment increased 14% ( 14 % ) from fiscal 2007 .\n- all revenue components within the segment experienced growth during fiscal 2008 .\n- license

## 4. 数据集清洗

使用 `clean_sharegpt_dataset.py` 对转换后的 SFT 文件进行统一清洗：
- 去掉空轮次 / 奇数轮次
- 去掉超长样本
- 限制单条样本的上下文长度


In [44]:
def sample_jsonl_records(path: Path, target_rows: int, seed: int = 42):
    with path.open('r', encoding='utf-8') as f:
        records = [line for line in f if line.strip()]
    if target_rows <= 0 or len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:target_rows])
    return [records[i] for i in idxs]

def run_clean(src: Path, dst: Path):
    clean_cmd = [
        'python', 'clean_sharegpt_dataset.py',
        '--source_file', str(src),
        '--output_file', str(dst),
        '--min_turns', '2',
        '--max_turns', '16',
        '--max_total_chars', '6000',
        '--max_single_value_chars', '2500',
    ]
    print(' '.join(clean_cmd))
    result = subprocess.run(clean_cmd, check=True, capture_output=True, text=True)
    print(result.stdout)

def run_audit_and_strict(clean_file: Path, audit_dir: Path, strict_file: Path):
    audit_cmd = [
        'python', 'audit_sharegpt_dirty_samples.py',
        '--input_file', str(clean_file),
        '--output_dir', str(audit_dir),
    ]
    print(' '.join(audit_cmd))
    audit_res = subprocess.run(audit_cmd, check=True, capture_output=True, text=True)
    print(audit_res.stdout)

    review_file = audit_dir / 'dirty_samples_review.jsonl'
    strict_cmd = [
        'python', 'filter_sharegpt_by_audit.py',
        '--input_file', str(clean_file),
        '--review_file', str(review_file),
        '--output_file', str(strict_file),
        '--mode', 'strict',
    ]
    print(' '.join(strict_cmd))
    strict_res = subprocess.run(strict_cmd, check=True, capture_output=True, text=True)
    print(strict_res.stdout)

# 1) SFT1 混合 + 清洗 + 严格过滤
with SFT1_MIXED_FILE.open('w', encoding='utf-8') as wf:
    for spec in SFT1_DATA_SPECS:
        path = SFT1_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
        sampled = sample_jsonl_records(path, spec['target_rows'], seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith('\n') else line + '\n')

run_clean(SFT1_MIXED_FILE, SFT1_CLEAN_FILE)
run_audit_and_strict(SFT1_CLEAN_FILE, SFT1_AUDIT_DIR, SFT1_STRICT_FILE)

# 2) SFT2: Fingpt主体 + 清洗后的SFT1 replay，再清洗 + 严格过滤
fingpt_rows = []
for spec in SFT2_DATA_SPECS:
    path = SFT2_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
    sampled = sample_jsonl_records(path, spec['target_rows'], seed=RANDOM_SEED)
    fingpt_rows.extend(sampled)

sft2_replay_rows = min(int(round(len(fingpt_rows) * SFT2_REPLAY_RATIO)), SFT2_REPLAY_MAX_ROWS)
replay_rows = sample_jsonl_records(SFT1_STRICT_FILE, sft2_replay_rows, seed=RANDOM_SEED)

with SFT2_MIXED_FILE.open('w', encoding='utf-8') as wf:
    for line in fingpt_rows:
        wf.write(line if line.endswith('\n') else line + '\n')
    for line in replay_rows:
        wf.write(line if line.endswith('\n') else line + '\n')

print(f"SFT2 fingpt主体样本: {len(fingpt_rows)}")
print(f"SFT2 replay样本(来自严格清洗SFT1): {len(replay_rows)}")

run_clean(SFT2_MIXED_FILE, SFT2_CLEAN_FILE)
run_audit_and_strict(SFT2_CLEAN_FILE, SFT2_AUDIT_DIR, SFT2_STRICT_FILE)

# Make train_file_dir-style folders (最终训练集使用 strict 版本)
(SFT1_DIR / 'train_sft1_clean_strict.jsonl').write_text(SFT1_STRICT_FILE.read_text(encoding='utf-8'), encoding='utf-8')
(SFT2_DIR / 'train_sft2_clean_strict.jsonl').write_text(SFT2_STRICT_FILE.read_text(encoding='utf-8'), encoding='utf-8')


python clean_sharegpt_dataset.py --source_file /root/autodl-tmp/data/financial_reasoning/mixed/train_sft1_mixed.jsonl --output_file /root/autodl-tmp/data/financial_reasoning/clean/train_sft1_clean.jsonl --min_turns 2 --max_turns 16 --max_total_chars 6000 --max_single_value_chars 2500
{
  "source_file": "/root/autodl-tmp/data/financial_reasoning/mixed/train_sft1_mixed.jsonl",
  "output_file": "/root/autodl-tmp/data/financial_reasoning/clean/train_sft1_clean.jsonl",
  "stats": {
    "raw_records": 12000,
    "drop_overlong_turn": 3306,
    "kept_records": 8694
  },
  "kept_turn_hist_top10": [
    [
      2,
      8694
    ]
  ],
  "kept_total_chars": {
    "min": 802,
    "p50": 2286,
    "p90": 2749,
    "p95": 2831,
    "p99": 2953,
    "max": 3064
  }
}

python audit_sharegpt_dirty_samples.py --input_file /root/autodl-tmp/data/financial_reasoning/clean/train_sft1_clean.jsonl --output_dir /root/autodl-tmp/data/financial_reasoning/clean/audit_sft1
{
  "input_file": "/root/autodl-tmp/dat

1775286

In [45]:
def line_count(path: Path) -> int:
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

sft2_fingpt_rows = sum(spec['target_rows'] for spec in SFT2_DATA_SPECS)
sft2_replay_target = min(int(round(sft2_fingpt_rows * SFT2_REPLAY_RATIO)), SFT2_REPLAY_MAX_ROWS)

report = {
    'disk_root': str(DISK_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'sft1_total_budget': SFT1_TOTAL_BUDGET,
    'sft2_extra_budget': SFT2_EXTRA_BUDGET,
    'sft2_replay_ratio': SFT2_REPLAY_RATIO,
    'sft2_replay_target_rows': sft2_replay_target,
    'dpo_total_budget': DPO_TOTAL_BUDGET,
    'sft1_allocations': [
        {'name': spec['name'], 'weight': spec['weight'], 'max_rows': spec['max_rows'], 'target_rows': spec['target_rows']}
        for spec in SFT1_DATA_SPECS
    ],
    'sft2_allocations': [
        {'name': spec['name'], 'weight': spec['weight'], 'max_rows': spec['max_rows'], 'target_rows': spec['target_rows']}
        for spec in SFT2_DATA_SPECS
    ],
    'sft1_raw_mix_rows': line_count(SFT1_MIXED_FILE),
    'sft1_clean_rows': line_count(SFT1_CLEAN_FILE),
    'sft1_strict_rows': line_count(SFT1_STRICT_FILE),
    'sft2_raw_mix_rows': line_count(SFT2_MIXED_FILE),
    'sft2_clean_rows': line_count(SFT2_CLEAN_FILE),
    'sft2_strict_rows': line_count(SFT2_STRICT_FILE),
}
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "disk_root": "/root/autodl-tmp",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning",
  "sft1_total_budget": 12000,
  "sft2_extra_budget": 2500,
  "sft2_replay_ratio": 0.2,
  "sft2_replay_target_rows": 342,
  "dpo_total_budget": 4000,
  "sft1_allocations": [
    {
      "name": "fingpt_convfinqa_train",
      "weight": 1.0,
      "max_rows": null,
      "target_rows": 7059
    },
    {
      "name": "finqa_train",
      "weight": 0.7,
      "max_rows": 6251,
      "target_rows": 4941
    }
  ],
  "sft2_allocations": [
    {
      "name": "fingpt_fineval_train",
      "weight": 1.0,
      "max_rows": 1060,
      "target_rows": 1060
    },
    {
      "name": "fingpt_fiqa_qa_train",
      "weight": 0.35,
      "max_rows": 1500,
      "target_rows": 648
    }
  ],
  "sft1_raw_mix_rows": 12000,
  "sft1_clean_rows": 8694,
  "sft1_strict_rows": 6769,
  "sft2_raw_mix_rows": 2045,
  "sft2_clean_rows": 1868,
  "sft2_strict_rows": 807
}


## 5. 数据集配比说明

当前策略：
- SFT-1：按预算与权重构建主干数据，然后清洗
- SFT-2：**以 Fingpt 数据为主体**，并按比例混入**清洗后的 SFT-1**作为 replay，再按与 SFT-1 相同规则清洗

默认参数：
- **SFT-1 总预算**：`SFT1_TOTAL_BUDGET = 12000`
- **SFT-2 Fingpt 预算**：`SFT2_EXTRA_BUDGET = 2500`
- **SFT-2 replay 比例**：`SFT2_REPLAY_RATIO = 0.2`
- **SFT-2 replay 上限**：`SFT2_REPLAY_MAX_ROWS = 3000`
- **DPO 总预算**：`DPO_TOTAL_BUDGET = 4000`
- **单数据集 DPO 上限**：`MAX_DPO_PER_DATASET = 2000`

说明：
- replay 明确来自 `SFT1_CLEAN_FILE`（清洗后的 SFT-1）
- SFT-2 不再直接拼接原始 ConvFinQA/FinQA 原始混合文件
- 这样可以降低阶段间重复训练偏置，同时保留必要的抗遗忘能力


In [38]:
def line_count(path: Path) -> int:
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

sft2_fingpt_rows = sum(spec['target_rows'] for spec in SFT2_DATA_SPECS)
sft2_replay_target = min(int(round(sft2_fingpt_rows * SFT2_REPLAY_RATIO)), SFT2_REPLAY_MAX_ROWS)

report = {
    'disk_root': str(DISK_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'sft1_total_budget': SFT1_TOTAL_BUDGET,
    'sft2_extra_budget': SFT2_EXTRA_BUDGET,
    'sft2_replay_ratio': SFT2_REPLAY_RATIO,
    'sft2_replay_target_rows': sft2_replay_target,
    'dpo_total_budget': DPO_TOTAL_BUDGET,
    'sft1_allocations': [
        {
            'name': spec['name'],
            'weight': spec['weight'],
            'max_rows': spec['max_rows'],
            'target_rows': spec['target_rows'],
        }
        for spec in SFT1_DATA_SPECS
    ],
    'sft2_allocations': [
        {
            'name': spec['name'],
            'weight': spec['weight'],
            'max_rows': spec['max_rows'],
            'target_rows': spec['target_rows'],
        }
        for spec in SFT2_DATA_SPECS
    ],
    'sft1_raw_mix_rows': line_count(SFT1_MIXED_FILE),
    'sft1_clean_rows': line_count(SFT1_CLEAN_FILE),
    'sft2_raw_mix_rows': line_count(SFT2_MIXED_FILE),
    'sft2_clean_rows': line_count(SFT2_CLEAN_FILE),
}
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "disk_root": "/root/autodl-tmp",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning",
  "sft1_total_budget": 12000,
  "sft2_extra_budget": 2500,
  "sft2_replay_ratio": 0.2,
  "sft2_replay_target_rows": 342,
  "dpo_total_budget": 4000,
  "sft1_allocations": [
    {
      "name": "fingpt_convfinqa_train",
      "weight": 1.0,
      "max_rows": null,
      "target_rows": 7059
    },
    {
      "name": "finqa_train",
      "weight": 0.7,
      "max_rows": 6251,
      "target_rows": 4941
    }
  ],
  "sft2_allocations": [
    {
      "name": "fingpt_fineval_train",
      "weight": 1.0,
      "max_rows": 1060,
      "target_rows": 1060
    },
    {
      "name": "fingpt_fiqa_qa_train",
      "weight": 0.35,
      "max_rows": 1500,
      "target_rows": 648
    }
  ],
  "sft1_raw_mix_rows": 12000,
  "sft1_clean_rows": 8694,
  "sft2_raw_mix_rows": 2045,
  "sft2_clean_rows": 1868
}


In [40]:
import json
from pathlib import Path
from collections import Counter

path = Path("/root/autodl-tmp/data/financial_reasoning/clean/train_sft1_clean.jsonl")

counter = Counter()
bad_rows = []

with path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            obj = json.loads(line)
        except Exception as e:
            print(f"[JSON 解析失败] 第{i}行: {e}")
            continue

        if "gold_ind" in obj:
            counter[f"gold_ind:{type(obj['gold_ind']).__name__}"] += 1
        else:
            counter["gold_ind:MISSING"] += 1

        if "gold_inds" in obj:
            counter[f"gold_inds:{type(obj['gold_inds']).__name__}"] += 1
        else:
            counter["gold_inds:MISSING"] += 1

        # 记录类型异常的行
        if "gold_ind" in obj and not isinstance(obj["gold_ind"], (str, dict, list, type(None))):
            bad_rows.append((i, type(obj["gold_ind"]).__name__, obj))

print("字段类型统计：")
for k, v in counter.items():
    print(k, v)

print("\n异常样本数：", len(bad_rows))
if bad_rows:
    print("\n第一条异常样本：")
    i, t, obj = bad_rows[0]
    print(f"第{i}行, 类型={t}")
    print(json.dumps(obj, ensure_ascii=False, indent=2)[:4000])

字段类型统计：
gold_ind:MISSING 8694
gold_inds:MISSING 8694

异常样本数： 0


In [41]:
import json
from pathlib import Path

path = Path("/root/autodl-tmp/data/financial_reasoning/clean/train_sft2_clean.jsonl")

for i, line in enumerate(path.open("r", encoding="utf-8"), 1):
    obj = json.loads(line)
    if "gold_ind" in obj and not isinstance(obj["gold_ind"], str):
        print(f"\n=== 第{i}行 gold_ind 类型异常: {type(obj['gold_ind']).__name__} ===")
        print(json.dumps(obj, ensure_ascii=False, indent=2)[:5000])
        break

## 6. SFT-1：主干推理训练

对应你的 TODO：
1. 数据集来源改为 `ConvFinQA & FinQA`
2. notebook 提供展示前几条案例的单元
3. 已清洗数据
4. 已统一转换为 MedicalGPT SFT 格式
5. 这里给出 `SFT-1` 训练命令


In [46]:
BASE_MODEL, SFT1_DIR, str(SFT1_OUT)

('/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct',
 PosixPath('/root/autodl-tmp/data/financial_reasoning/clean/sft1_dir_strict'),
 '/root/autodl-tmp/outputs/financial_reasoning/sft1_lora')

`tensorboard --logdir=/root/MedicalGPT/outputs/financial_reasoning_sft1_lora/runs`

In [ ]:
sft1_cmd = [
    'python', 'supervised_finetuning.py',
    '--model_name_or_path', BASE_MODEL,
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT1_DIR),

    '--validation_split_percentage', '1',
    '--do_eval',
    '--eval_steps', '100',
    '--evaluation_strategy', 'steps',

    '--do_train',
    '--use_peft',

    '--num_train_epochs', '2',

    '--per_device_train_batch_size', '1',
    '--max_grad_norm', '1.0',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',

    '--warmup_ratio', '0.05',
    '--weight_decay', '0.05',
    '--learning_rate', '1e-5',

    '--logging_steps', '10',
    '--save_steps', '200',

    '--logging_first_step', 'True',
    '--report_to', 'tensorboard',

    '--model_max_length', '1024',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'bfloat16',

    '--device_map', 'auto',
    
    '--output_dir', str(SFT1_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(sft1_cmd))


python supervised_finetuning.py --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --train_file_dir data/financial_reasoning/clean/sft1_dir --validation_split_percentage 1 --do_eval --eval_steps 100 --evaluation_strategy steps --do_train --use_peft --num_train_epochs 2 --per_device_train_batch_size 1 --max_grad_norm 1.0 --gradient_accumulation_steps 16 --gradient_checkpointing True --warmup_ratio 0.05 --weight_decay 0.05 --learning_rate 1e-5 --logging_steps 10 --save_steps 200 --logging_first_step True --report_to tensorboard --logging_dir outputs/tensorboard/financial_reasoning/sft1 --model_max_length 1024 --target_modules all --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 --torch_dtype bf16 --device_map auto --output_dir outputs/financial_reasoning_sft1_lora --template_name qwen


## 7. SFT-2：中文补强

在 `SFT-1` 完成后：
- 先 merge `SFT-1 LoRA`
- 再用 `fingpt-fineval + 少量 fiqa_qa` 做二阶段 SFT


In [57]:
SFT1_MERGED_OUT

PosixPath('/root/autodl-tmp/outputs/financial_reasoning/sft1_merged')

In [58]:
SFT1_OUT = "/root/autodl-tmp/outputs/financial_reasoning/sft1_lora_strict_scratch"

In [50]:
merge_sft1_cmd = [
    'python', 'merge_peft_adapter.py',
    '--base_model', BASE_MODEL,
    '--tokenizer_path', BASE_MODEL,
    '--lora_model', str(SFT1_OUT),
    '--output_dir', str(SFT1_MERGED_OUT),
]
print(' '.join(merge_sft1_cmd))
# subprocess.run(merge_sft1_cmd, check=True)

python merge_peft_adapter.py --base_model /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --lora_model /root/autodl-tmp/outputs/financial_reasoning_sft1_lora_strict_scratch --output_dir /root/autodl-tmp/outputs/financial_reasoning/sft1_merged


In [51]:
str(SFT2_DIR), str(SFT2_OUT), str(TB_LOG_DIR / 'sft2')

('/root/autodl-tmp/data/financial_reasoning/clean/sft2_dir_strict',
 '/root/autodl-tmp/outputs/financial_reasoning/sft2_lora',
 '/root/autodl-tmp/outputs/financial_reasoning/tensorboard/sft2')

In [59]:
sft2_cmd = [
    'python', 'supervised_finetuning.py',
    '--model_name_or_path', str(SFT1_MERGED_OUT),
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT2_DIR),

    '--do_train',
    '--use_peft',
    '--num_train_epochs', '2',

    '--per_device_train_batch_size', '1',
    '--max_grad_norm', '0.5',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',

    '--learning_rate', '5e-6',

    '--warmup_ratio', '0.05',
    '--weight_decay', '0.05',

    '--logging_steps', '10',
    '--save_steps', '200',
    '--save_total_limit', '2',
    '--logging_first_step', 'True',

    '--report_to', 'tensorboard',
    '--logging_dir', str(TB_LOG_DIR / 'sft2'),
    '--model_max_length', '1024',
    '--target_modules', 'q_proj,k_proj,v_proj,o_proj',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'float16',
    '--device_map', 'auto',
    '--output_dir', str(SFT2_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(sft2_cmd))
# subprocess.run(sft2_cmd, check=True)


python supervised_finetuning.py --model_name_or_path /root/autodl-tmp/outputs/financial_reasoning/sft1_merged --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --train_file_dir /root/autodl-tmp/data/financial_reasoning/clean/sft2_dir_strict --do_train --use_peft --num_train_epochs 2 --per_device_train_batch_size 1 --max_grad_norm 0.5 --gradient_accumulation_steps 16 --gradient_checkpointing True --learning_rate 5e-6 --warmup_ratio 0.05 --weight_decay 0.05 --logging_steps 10 --save_steps 200 --save_total_limit 2 --logging_first_step True --report_to tensorboard --logging_dir /root/autodl-tmp/outputs/financial_reasoning/tensorboard/sft2 --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 --torch_dtype float16 --device_map auto --output_dir /root/autodl-tmp/outputs/financial_reasoning/sft2_lora --template_name qwen


In [62]:
!python merge_peft_adapter.py \
  --base_model /root/autodl-tmp/outputs/financial_reasoning/sft1_merged \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --lora_model /root/autodl-tmp/outputs/financial_reasoning/sft2_lora \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning/sft2_merged


Namespace(base_model='/root/autodl-tmp/outputs/financial_reasoning/sft1_merged', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', lora_model='/root/autodl-tmp/outputs/financial_reasoning/sft2_lora', resize_emb=False, output_dir='/root/autodl-tmp/outputs/financial_reasoning/sft2_merged', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/outputs/financial_reasoning/sft1_merged
LoRA model: /root/autodl-tmp/outputs/financial_reasoning/sft2_lora
Loading LoRA for causal language model
Loading weights: 100%|████████████████████████| 339/339 [00:03<00:00, 97.83it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100%|███████████████████████| 2/2 [00:23<00:00, 11.86s/it]
Done! model saved to /root/autodl-tmp/outputs/financial_reasoning/sft2_merged


In [ ]:
!python inference.py \
  --base_model /root/autodl-tmp/outputs/financial_reasoning/sft2_merged \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --single_tune  --interactive

Namespace(base_model='/root/autodl-tmp/outputs/financial_reasoning/sft2_merged', lora_model='', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', system_prompt='', stop_str='', repetition_penalty=1.0, max_new_tokens=512, data_file=None, interactive=False, single_tune=True, temperature=0.7, output_file='./predictions_result.jsonl', eval_batch_size=4, resize_emb=False, load_in_8bit=False, load_in_4bit=False)
Loading weights: 100%|███████████████████████| 339/339 [00:03<00:00, 101.17it/s]
Qwen2Tokenizer(name_or_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, spe

In [ ]:
!python inference.py \
  --base_model /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --single_tune  --interactive

Namespace(base_model='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', lora_model='', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', system_prompt='', stop_str='', repetition_penalty=1.0, max_new_tokens=512, data_file=None, interactive=False, single_tune=True, temperature=0.7, output_file='./predictions_result.jsonl', eval_batch_size=4, resize_emb=False, load_in_8bit=False, load_in_4bit=False)
Loading weights: 100%|████████████████████████| 339/339 [00:03<00:00, 95.85it/s]
Qwen2Tokenizer(name_or_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=T

In [ ]:
!python inference.py \
  --base_model /root/autodl-tmp/outputs/financial_reasoning/sft1_merged \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --single_tune --interactive



Namespace(base_model='/root/autodl-tmp/outputs/financial_reasoning/sft1_merged', lora_model='', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', system_prompt='', stop_str='', repetition_penalty=1.0, max_new_tokens=512, data_file=None, interactive=False, single_tune=True, temperature=0.7, output_file='./predictions_result.jsonl', eval_batch_size=4, resize_emb=False, load_in_8bit=False, load_in_4bit=False)
Loading weights: 100%|████████████████████████| 339/339 [00:03<00:00, 94.13it/s]
Qwen2Tokenizer(name_or_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, spe

## 8. 轻量 DPO（可选）

轻量 DPO 目标：
- 优化表达质量
- 保持结构完整
- 减少废话
- 不让偏好训练覆盖主干 reasoning 能力


In [ ]:
with DPO_MIXED_FILE.open('w', encoding='utf-8') as wf:
    all_specs = SFT1_DATA_SPECS + SFT2_DATA_SPECS
    total_source_target = sum(spec['target_rows'] for spec in all_specs if spec['target_rows'] > 0)
    for spec in all_specs:
        path = DPO_DIR / f"{spec['name']}_dpo.jsonl"
        if total_source_target > 0:
            proportional_target = int(round(DPO_TOTAL_BUDGET * spec['target_rows'] / total_source_target))
        else:
            proportional_target = 0
        dpo_target = min(MAX_DPO_PER_DATASET, spec.get('max_rows') or 10**9, proportional_target)
        sampled = sample_jsonl_records(path, dpo_target, seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith('
') else line + '
')

(DPO_TRAIN_DIR / DPO_MIXED_FILE.name).write_text(DPO_MIXED_FILE.read_text(encoding='utf-8'), encoding='utf-8')

merge_sft2_cmd = [
    'python', 'merge_peft_adapter.py',
    '--base_model', BASE_MODEL,
    '--tokenizer_path', BASE_MODEL,
    '--lora_model', str(SFT2_OUT),
    '--output_dir', str(SFT2_MERGED_OUT),
]
print(' '.join(merge_sft2_cmd))
# subprocess.run(merge_sft2_cmd, check=True)

dpo_cmd = [
    'python', 'dpo_training.py',
    '--model_name_or_path', str(SFT2_MERGED_OUT),
    '--tokenizer_name_or_path', BASE_MODEL,
    '--template_name', TEMPLATE_NAME,
    '--train_file_dir', str(DPO_TRAIN_DIR),
    '--validation_split_percentage', '1',
    '--do_train',
    '--use_peft', 'True',
    '--per_device_train_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',
    '--learning_rate', '5e-7',
    '--max_steps', '200',
    '--max_source_length', '1024',
    '--max_target_length', '512',
    '--logging_steps', '10',
    '--save_steps', '200',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'float16',
    '--device_map', 'auto',
    '--output_dir', str(DPO_OUT),
]
print(' '.join(dpo_cmd))
# subprocess.run(dpo_cmd, check=True)


## 9. GRPO reward 设计（推荐）

针对金融 reasoning，推荐 3 类 reward：
1. **格式奖励**：是否满足结构化输出格式
2. **程序一致性奖励**：是否包含与 gold program 一致的关键操作
3. **答案正确性奖励**：最终答案是否与 gold answer 一致 / 数值接近

这里给出 reward 设计原型，后续可以把它接到 `grpo_training.py`。


In [ ]:
import re

SECTION_PATTERNS = {
    'analysis': r'问题分析：',
    'program': r'推理程序：',
    'answer': r'最终答案：',
}


def reasoning_format_reward(text: str) -> float:
    score = 0.0
    for pattern in SECTION_PATTERNS.values():
        if re.search(pattern, text):
            score += 1.0
    return score / len(SECTION_PATTERNS)


def extract_final_answer(text: str) -> str:
    m = re.search(r'最终答案：\s*(.+)', text)
    return m.group(1).strip() if m else text.strip()


def normalize_number(text: str):
    m = re.search(r'-?\d+(?:,\d{3})*(?:\.\d+)?', text.replace(',', ''))
    return float(m.group(0)) if m else None


def answer_correctness_reward(pred: str, gold: str, tol: float = 1e-4) -> float:
    pred_num = normalize_number(extract_final_answer(pred))
    gold_num = normalize_number(gold)
    if pred_num is not None and gold_num is not None:
        return 1.0 if abs(pred_num - gold_num) <= tol else 0.0
    return 1.0 if extract_final_answer(pred).strip() == gold.strip() else 0.0


def program_consistency_reward(pred: str, gold_program: str) -> float:
    if not gold_program:
        return 0.0
    pred_program = ''
    m = re.search(r'推理程序：\s*(.+)', pred)
    if m:
        pred_program = m.group(1).strip()
    gold_ops = [op for op in ['add', 'subtract', 'multiply', 'divide', 'greater', 'table_max', 'table_min'] if op in gold_program]
    if not gold_ops:
        return 0.0
    hit = sum(1 for op in gold_ops if op in pred_program)
    return hit / len(gold_ops)

print('reward design ready')


## 10. benchmark 评估

在 `SFT-2` 训练并 merge 完成后，直接评估三组模型：
- `base`：基座模型
- `sft1`：`SFT1` merge 后模型
- `sft2`：`SFT1 + SFT2` merge 后模型

当前 benchmark 包含：
- `ConvFinQA test`：英文金融数值推理 / 表格理解
- `fingpt-fineval test`：中文金融知识 / 考试型推理
- `CFLUE` 小规模外部任务集：中文金融泛化


In [ ]:
BENCHMARK_OUTPUT_DIR = OUTPUT_ROOT / "benchmarks" / "sft2_compare"
BENCHMARK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONVFINQA_TEST_FILE = RAW_CACHE_DIR / "convfinqa_turn" / "test_turn.json"
FINEVAL_TEST_LOCAL_FILE = ""

# 按需填写本地 CFLUE 子任务文件；不存在的文件会自动跳过
CFLUE_TASK_FILES = {
    # "bank": str(RAW_CACHE_DIR / "cflue" / "bank.jsonl"),
    # "securities": str(RAW_CACHE_DIR / "cflue" / "securities.jsonl"),
    # "fund": str(RAW_CACHE_DIR / "cflue" / "fund.jsonl"),
}

EVAL_MAX_SAMPLES = {
    "convfinqa": 0,
    "fineval": 0,
    "cflue_per_task": 0,
}

print("BENCHMARK_OUTPUT_DIR:", BENCHMARK_OUTPUT_DIR)
print("CONVFINQA_TEST_FILE:", CONVFINQA_TEST_FILE)
print("CFLUE_TASK_FILES:", json.dumps(CFLUE_TASK_FILES, ensure_ascii=False, indent=2))


In [ ]:
import subprocess
from pathlib import Path

eval_cmd = [
    "python", "evaluate_financial_benchmarks.py",
    "--tokenizer_path", BASE_MODEL,
    "--model_entry", f"base={BASE_MODEL}",
    "--model_entry", f"sft1={SFT1_MERGED_OUT}",
    "--model_entry", f"sft2={SFT2_MERGED_OUT}",
    "--convfinqa_test_file", str(CONVFINQA_TEST_FILE),
    "--convfinqa_max_samples", str(EVAL_MAX_SAMPLES["convfinqa"]),
    "--fineval_dataset_name", "FinGPT/fingpt-fineval",
    "--fineval_split", "test",
    "--fineval_max_samples", str(EVAL_MAX_SAMPLES["fineval"]),
    "--cflue_max_samples_per_task", str(EVAL_MAX_SAMPLES["cflue_per_task"]),
    "--output_dir", str(BENCHMARK_OUTPUT_DIR),
]

if FINEVAL_TEST_LOCAL_FILE:
    eval_cmd.extend(["--fineval_local_file", FINEVAL_TEST_LOCAL_FILE])

for task_name, task_path in CFLUE_TASK_FILES.items():
    if Path(task_path).exists():
        eval_cmd.extend(["--cflue_task_file", f"{task_name}={task_path}"])
    else:
        print(f"[skip] missing CFLUE file: {task_path}")

print(" ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)
